# 기존 코드
<details>
<summary>코드 보기 (여기를 클릭하세요)</summary>

```python
#제품명 입력
product_name = 'product_name'

# 몇 페이지까지 크롤링할지 설정
max_pages = 20
current_page = 1

#크롤링 사이트 - 리뷰에서 see more review
url = 'url'

# 리뷰 페이지 열기
driver.get(url)

# CAPTCHA 및 로그인 인증 통과 대기

input("CAPTCHA를 통과한 후 엔터를 누르세요...")

# 결과 저장 리스트
all_reviews = []


while current_page <= max_pages:
    print(f"페이지 {current_page} 수집 중...")

    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, "//div[starts-with(@id, 'customer_review-')]"))
        )
    except:
        print("리뷰 블럭 로딩 실패, 다음 페이지로")
        break

    review_blocks = driver.find_elements(By.XPATH, "//div[starts-with(@id, 'customer_review-')]")
    print(f"  → {len(review_blocks)}개 리뷰 블럭 탐지됨")

    for block in review_blocks:
        try:
            author = block.find_element(By.XPATH, ".//div[@class='a-profile-content']/span").text
        except:
            author = ""

        try:
            rating_text = block.find_element(By.XPATH, ".//i[@data-hook='review-star-rating']").get_attribute("innerText")
            rating = float(rating_text.split()[0])
            rating = int(rating)
        except:
            rating = None

        try:
            review_info = block.find_element(By.XPATH, ".//span[contains(text(), 'Reviewed in')]").text
            if " on " in review_info:
                location = review_info.split("Reviewed in ")[1].split(" on ")[0].strip()
                date = review_info.split(" on ")[1].strip()
            else:
                location = ""
                date = ""
        except:
            location = ""
            date = ""

        try:
            content = block.find_element(By.XPATH, ".//span[@data-hook='review-body']//span").text
        except:
            content = ""

        all_reviews.append({
            '작성자': author,
            '별점': rating,
            '작성지': location,
            '작성일': date,
            '리뷰 내용': content
        })

    try:
        next_button = driver.find_element(By.XPATH, "//li[@class='a-last']/a")
        next_button.click()
        time.sleep(5)
        current_page += 1
    except:
        print("다음 페이지 없음 또는 버튼 클릭 불가")
        break

# 결과 저장
with open(f"amazon_reviews_{product_name}.csv", "w", newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['작성자', '별점', '작성지', '작성일', '리뷰 내용'])
    writer.writeheader()
    writer.writerows(all_reviews)

print(f"총 {len(all_reviews)}개 리뷰 수집 완료")
driver.quit()
```

---

# Amazon 리뷰 크롤링 개선 문서

## 1. 발견된 문제점

기존 크롤링 시스템에서 두 가지 주요 문제가 확인되었습니다.

**문제 1: 리뷰 개수 제한**
- 100개 이상의 리뷰가 있는 제품에서 100개를 초과하는 리뷰는 크롤링이 불가능했습니다.

**문제 2: 지역 제한**
- 검색 리전을 미국으로 설정한 경우, 미국 이외 지역에서 작성된 리뷰는 크롤링되지 않았습니다.

---

## 2. 문제 원인 분석

### 2.1 리뷰 개수 제한 문제
- Amazon은 기본적으로 100개 이상의 리뷰를 한 번에 제공하지 않습니다. 
- 사용자는 '상위 100개 리뷰' 또는 '최신 100개 리뷰'만 조회할 수 있으며, 이로 인해 대량의 리뷰 데이터 수집에 제약이 발생했습니다.

### 2.2 지역별 리뷰 크롤링 문제
- Amazon은 사용자의 지역과 동일한 지역에서 작성된 리뷰와 다른 지역 리뷰에 대해 서로 다른 HTML 구조를 사용합니다. 
- 기존 코드는 미국 지역의 HTML 태그 구조만을 기준으로 작성되어, 다른 지역의 리뷰는 파싱할 수 없었습니다.

---

## 3. 해결 방안

### 3.1 리뷰 개수 제한 극복
Amazon의 리뷰 검색 기능을 활용하여 더 많은 리뷰를 수집할 수 있도록 개선했습니다.

**검색 기능의 특징**
- 구(phrase) 형태로 검색 시, 정확히 일치하는 경우만 검색됩니다.
- 따라서 복수의 단어를 동시에 검색하는 방식은 효과적이지 않습니다.

**적용 방법**
- 개별 단어를 하나씩 검색창에 입력하여 다양한 리뷰 세트를 수집합니다.
- 여러 키워드로 반복 검색하여 100개 제한을 우회적으로 극복합니다.

**개별 키워드 선택 방법**
1. 단어 빈도 분석(Frequency-based)
2. TF-IDF (Term Frequency-Inverse Document Frequency)

### 3.2 지역별 HTML 구조 대응
코드 내 HTML 태그 참조 방식을 개선하여 다양한 지역의 리뷰를 크롤링할 수 있도록 수정했습니다.

**테스트 사례**
- 제품: 삼양 불닭볶음면 (spicy_chicken_ramen)
- 결과: 
    - 100개 이상의 데이터 수집 성공
    - 미국 외 지역에서 작성된 리뷰도 성공적으로 크롤링 확인
---

## 4. 데이터 분석을 위한 추가 개선사항

크롤링 시스템을 단순 수집을 넘어 분석 가능한 형태로 고도화했습니다.

**개선 1: 구매 옵션 정보 수집**
- 제품의 구매 옵션 데이터를 별도 컬럼으로 추가하여, 옵션별 리뷰 분석이 가능하도록 개선했습니다.

**개선 2: 재구매 관련 키워드 자동 검색**
- 리뷰 업데이트, 재구매, 재구매 의사 등을 표현하는 키워드를 고정 검색어로 추가하여 고객 충성도 관련 데이터를 체계적으로 수집합니다.

**개선 3: 불용어 설정**
- 제품명은 대부분의 리뷰에서 반복적으로 등장하므로 불용어로 지정하여, 텍스트 분석 시 의미 있는 키워드 추출이 가능하도록 설정했습니다.

---

## 5. 결론

이번 개선을 통해 Amazon 리뷰 크롤링 시스템의 데이터 수집 범위와 품질이 크게 향상되었습니다. 리뷰 개수 제한과 지역 제한이라는 두 가지 주요 장애물을 극복했으며, 수집된 데이터의 분석 활용도를 높이는 추가 기능도 구현했습니다.

In [31]:
import pandas as pd
import re
import time
import os
import random
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS

In [ ]:
# 이렇게 출력한 단어들을 바탕으로 해서 다시 검새하는 코드 작성
def start_driver():
    options = Options()
    options.add_argument('--start-maximized')
    driver = webdriver.Chrome(options=options)
    return driver

def collect_reviews_from_current_page(driver, keyword = "Initial"):
    """
    현재의 페이지의 리뷰 블록들을 파싱하여 리스트로
    주석보고 파싱 되는지 확인
    """
    reviews_data = []
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, "//div[starts-with(@id, 'customer_review-')]"))
        )
    except:
        return []

    review_blocks = driver.find_elements(By.XPATH, "//div[starts-with(@id, 'customer_review-')]")
    
    for block in review_blocks:
        # 작성자
        try:
            author = block.find_element(By.XPATH, ".//div[@class='a-profile-content']/span").text
        except: author = ""

        # 별점
        try:
            rating_text = block.find_element(By.XPATH, ".//i[@data-hook='review-star-rating']").get_attribute("innerText")
            rating = float(rating_text.split()[0])
        except: rating = None

        # 작성지/작성일
        try:
            review_info = block.find_element(By.XPATH, ".//span[contains(text(), 'Reviewed in')]").text
            if " on " in review_info:
                location = review_info.split("Reviewed in ")[1].split(" on ")[0].strip()
                date = review_info.split(" on ")[1].strip()
            else:
                location = ""
                date = ""
        except:
            location = ""; date = ""
        
        # 옵션
        try:
            opt_elem = block.find_element(By.XPATH, ".//a[@data-hook='format-strip']")
            
            raw_html = opt_elem.get_attribute('innerHTML')
            
            processed_html = re.sub(r'<i.*?</i>', ' | ', raw_html)
            
            clean_text = re.sub(r'<[^>]+>', '', processed_html)
            
            option = " ".join(clean_text.split())
            
        except:
            option = ""

        # 리뷰 내용
        try:
            content = block.find_element(By.XPATH, ".//span[@data-hook='review-body']//span").text
        except: content = ""

        reviews_data.append({
            '검색어': keyword,
            '작성자': author,
            '별점': rating,
            '작성지': location,
            '작성일': date,
            '옵션': option,
            '리뷰 내용': content
        })
        
    return reviews_data

In [33]:
def get_custom_stopwords(words_list = None):
    stop_words = set(ENGLISH_STOP_WORDS)
    
    if words_list:
        stop_words.update(words_list)
    return list(stop_words)


def get_combined_keywords(text_data, top_n = 15, words_list = None):
    custom_stopwords = get_custom_stopwords(words_list)
    if hasattr(text_data, 'dropna'):
        text_data = text_data.dropna().tolist()
    
    # countvector를 활용한 빈도 계산(단순히 가장 많이 등장한 단어 추출)  
    count_vectorizer = CountVectorizer(stop_words=custom_stopwords)
    X_counts = count_vectorizer.fit_transform(text_data)
    sum_words = X_counts.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in count_vectorizer.vocabulary_.items()]
    top_words = [w[0] for w in sorted(words_freq, key=lambda x: x[1], reverse=True)[:top_n]]
    
    # TF-IDF 분석 (특정 리뷰들에서 중요하게 쓰인 단어)
    tfidf_vectorizer = TfidfVectorizer(stop_words=custom_stopwords)
    X_tfidf = tfidf_vectorizer.fit_transform(text_data)
    sum_tfidf = X_tfidf.sum(axis=0)
    words_tfidf = [(word, sum_tfidf[0, idx]) for word, idx in tfidf_vectorizer.vocabulary_.items()]
    top_tfidf = [w[0] for w in sorted(words_tfidf, key=lambda x: x[1], reverse=True)[:top_n]]
    
    # set을 통해 중복되는 키워드 제거
    keyword = set(top_words)
    keyword.update(top_tfidf)
    print(keyword)
    return list(keyword)

In [34]:
def refined_wordlist(wordlist:list):
    """
    아마존 단어 검색 방식(불필요한 재검색을 방지)에 맞게 재검색 단어들을 재정재하는 함수
    """
    result = []
    wordlist = sorted(wordlist, key=lambda x : len(x))
    check_list = sorted(('s', 'es', 'ed', 'ing'), key=len, reverse=True)
    
    for word in wordlist:
        str_word = str(word)
        matched_suffix = None
        
        for suffix in check_list:
            if str_word.endswith(suffix):
                matched_suffix = suffix
                break
        
        if matched_suffix:
            root = str_word[:-len(matched_suffix)]
            
            if root not in result:
                result.append(str_word)
        else:
            result.append(str_word)
    return result

In [35]:
def human_delay(min_sec=2, max_sec=5):
    time.sleep(random.uniform(min_sec, max_sec))
    
def human_scroll(driver):
    """페이지를 자연스럽게 읽는 것처럼 스크롤함"""
    total_height = driver.execute_script("return document.body.scrollHeight")
    current_pos = 0
    while current_pos < total_height:
        step = random.randint(300, 600) # 한 번에 내릴 픽셀
        current_pos += step
        driver.execute_script(f"window.scrollTo(0, {current_pos});")
        time.sleep(random.uniform(0.3, 0.8)) # 잠깐 멈춰서 읽는 척
        if current_pos > 2000: break # 너무 길면 중간에 끊기
    human_delay(1, 2)
    

In [36]:
def crawl_amazon_keyword(driver, keyword, max_pages=5):
    """
    특정 키워드를 검색창에 입력하고 결과를 크롤링하는 함수
    """
    collected_data = []
    
    print(f"\n=== 키워드 검색 시작: '{keyword}' ===")
    
    try:
        search_box = WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.XPATH, '//*[@id="filterByKeywordTextBox"]'))
        )
        ActionChains(driver).move_to_element(search_box).click().perform()
        search_box.clear()
        search_box.send_keys(keyword)
        
        search_button = driver.find_element(By.CSS_SELECTOR, '#a-autoid-1 > span > input')
        search_button.click()
        time.sleep(3)
        
    except Exception as e:
        print(f"검색창 조작 실패 ({keyword}): {e}")
        return []

    current_page = 1
    while current_page <= max_pages:
        print(f"  [Keyword: {keyword}] Page {current_page}...")
        
        # 페이지 로딩 후 대기
        human_scroll(driver)
        
        page_data = collect_reviews_from_current_page(driver, keyword)
        if not page_data:
            print("  → 리뷰 없음, 루프 종료")
            break
            
        collected_data.extend(page_data)

        try:
            next_button = driver.find_element(By.XPATH, "//li[@class='a-last']/a")
            if "a-disabled" in next_button.find_element(By.XPATH, "..").get_attribute("class"):
                break
            ActionChains(driver).move_to_element(next_button).perform()
            human_delay(1, 2)
            next_button.click()
            human_delay(5, 9)
            current_page += 1
        except:
            break
            
    return collected_data

In [27]:
def split_option(df, col_name = "옵션"):
    """
    데이터 프레임의 옵션 컬럼("key : value | key : value" 형태의 문자열)을 파싱하여 새로운 컬럼 생성
    
    Args:
        df : 데이터 프레임
        col_name : 옵션 컬럼
    
    Return:
        result_df : 옵션 컬럼이 변형된 데이터 프레임
    """
    
    if col_name not in df.columns:
        return df
    
    def parse_row(row_str):
        if not isinstance(row_str, str):
            return {}
        
        data = {}
        splited_option = row_str.split('|')
        
        for opt in splited_option:
            if ':' in opt:
                key, val = opt.split(':', 1)
                
                key = key.strip().lower()
                val = val.strip()
                data[key] = val
        return data
    
    split_col = df[col_name].apply(parse_row)
    
    option_df = pd.json_normalize(split_col)
    
    option_df.index = df.index
    result_df = pd.concat([df, option_df], axis=1)
    # result_df.drop(columns=[col_name], inplace=True)
    
    return result_df

In [ ]:
# 제품 이름 : 아마존에 등록된 제품 이름
PRODUCT_NAME = 'The Heaven & Earth Grocery Store - 2023'
# URL 주소는 전체 리뷰를 볼수 있는 페이지를 선택
INITIAL_URL = 'https://www.amazon.com/product-reviews/B0BPNP7YQB/ref=cm_cr_dp_d_show_all_btm?ie=UTF8&reviewerType=all_reviews'

# 페이지 설정
INITIAL_MAX_PAGES = 10      # 초기에 긁어올 페이지 수
KEYWORD_MAX_PAGES = 10      # 키워드별로 긁어올 페이지 수

# 불용어 및 키워드 설정
TOP_N_KEYWORDS = 10 # 상위 몇개의 단어를 선택할지(추가적으로 검색할 단어 선정 과정에서)

# 첫번째 크롤링을 통해 키워드들을 파악한 후에 삭제하고자하는 단어들을 추가
MY_STOPWORDS = ['don', 'like', 've']

# 제품명에 들어가는 단어들은 불용어 처리를 하여 제거
DEL_TITLE = PRODUCT_NAME.lower().split()
MY_STOPWORDS += DEL_TITLE

# 일반적으로 추가해서 보려는 단어
#   1. 리뷰의 변화를 나타내는 단어 : update, edit
#   2. 재구매를 나타내는 표현 : 'second', 'again', 'reorder', 'another' 등
SELF_SEARCH_WORDS = ['conflict']

In [38]:
# 드라이버 시작
driver = start_driver()
driver.get(INITIAL_URL)

# CAPTCHA 처리 + 로그인 등 페이지로 이동하는 작업 수행
print("브라우저에서 CAPTCHA를 해결해주세요.")
input("리뷰 페이지가 보이면 엔터를 누르세요...")

# 최초 데이터 수집
print("최초 리뷰 데이터 수집 중...")
initial_reviews = []
current_page = 1

while current_page <= INITIAL_MAX_PAGES:
    print(f"  Initial Page {current_page}...")
    page_data = collect_reviews_from_current_page(driver, keyword="Initial_Crawl")
    initial_reviews.extend(page_data)
    
    # 다음 페이지 이동
    try:
        next_button = driver.find_element(By.XPATH, "//li[@class='a-last']/a")
        next_button.click()
        human_delay(2, 5)
        current_page += 1
    except:
        break

df_initial = pd.DataFrame(initial_reviews)
print(f"초기 수집 완료: {len(df_initial)}개 리뷰")


# 데이터 분석 및 추가 키워드 추출
print("\n텍스트 분석 및 추가 키워드 추출 중...")

extracted_keywords = []

if not df_initial.empty:
    extracted_keywords = get_combined_keywords(
        text_data=df_initial['리뷰 내용'],
        top_n=TOP_N_KEYWORDS,
        words_list=MY_STOPWORDS
    )
    print(f"추출된 키워드: {extracted_keywords}")

if len(extracted_keywords) > 0:
    refined_wordlist(extracted_keywords)

if SELF_SEARCH_WORDS:
    print(f'사용자 지정 단어 추가 : {SELF_SEARCH_WORDS}')
    extracted_keywords.extend(SELF_SEARCH_WORDS)

extracted_keywords = list(set(extracted_keywords))
print(f"최종 검색할 키워드 리스트: {extracted_keywords}")

# 추가 키워드로 2차 크롤링 (Concat을 위해 List에 모음)
print("\n추출된 키워드로 상세 크롤링 시작...")
additional_reviews = []

# 각 키워드별 크롤링 수행
for keyword in extracted_keywords:
    kw_data = crawl_amazon_keyword(driver, keyword, max_pages=KEYWORD_MAX_PAGES)
    additional_reviews.extend(kw_data)

driver.quit()

# 데이터 병합(최초 데이터 + 키워드 데이터), 중복 제거 및 최종 저장
print("\nSTEP 4: 데이터 병합 및 저장 중...")

df_additional = pd.DataFrame(additional_reviews)

if not df_additional.empty:
    print(f"추가 수집된 데이터: {len(df_additional)}개")
    final_df = pd.concat([df_initial, df_additional], ignore_index=True)
else:
    print("추가 수집된 데이터가 없습니다.")
    final_df = df_initial

initial_len = len(final_df)
final_df.drop_duplicates(subset=['작성자', '리뷰 내용'], keep='first', inplace=True)
final_df = split_option(final_df, '옵션')
print(f"중복 제거: {initial_len - len(final_df)}건 삭제됨.")

os.makedirs('data', exist_ok=True)

output_filename = f"amazon_reviews_{PRODUCT_NAME}_final.csv"
final_df.to_csv(output_filename, index=False, encoding='utf-8-sig')

print(f"\n총 {len(final_df)}개의 리뷰가 '{output_filename}'에 저장되었습니다.")

브라우저에서 CAPTCHA를 해결해주세요.
최초 리뷰 데이터 수집 중...
  Initial Page 1...
초기 수집 완료: 8개 리뷰

텍스트 분석 및 추가 키워드 추출 중...
{'inhabitants', 'story', 'lives', 'mcbride', 'town', 'characters', 'way', 'chona', 'black', 'book', 'immigrants', 'good', 'read', 'life', 'author', 'hill', 'real', 'people'}
추출된 키워드: ['inhabitants', 'story', 'lives', 'mcbride', 'town', 'characters', 'way', 'chona', 'black', 'book', 'immigrants', 'good', 'read', 'life', 'author', 'hill', 'real', 'people']
사용자 지정 단어 추가 : ['conflict']
최종 검색할 키워드 리스트: ['chona', 'black', 'good', 'read', 'hill', 'story', 'lives', 'conflict', 'people', 'inhabitants', 'town', 'way', 'immigrants', 'author', 'mcbride', 'characters', 'book', 'life', 'real']

추출된 키워드로 상세 크롤링 시작...

=== 키워드 검색 시작: 'chona' ===
검색창 조작 실패 (chona): Message: 
Stacktrace:
0   chromedriver                        0x0000000103087928 cxxbridge1$str$ptr + 3098388
1   chromedriver                        0x000000010307fc04 cxxbridge1$str$ptr + 3066352
2   chromedriver                        0x

In [20]:
# # 설정 변수
# PRODUCT_NAME = 'NutraChamps Korean Red Panax Ginseng Capsules'
# # URL 주소는 전체 리뷰를 볼수 있는 페이지를 선택
# INITIAL_URL = 'https://www.amazon.com/product-reviews/B01MECVTDY/ref=cm_cr_dp_d_show_all_btm?ie=UTF8&reviewerType=all_reviews'

# # 페이지 설정
# INITIAL_MAX_PAGES = 10      # 초기에 긁어올 페이지 수
# KEYWORD_MAX_PAGES = 4      # 키워드별로 긁어올 페이지 수

# # 불용어 및 키워드 설정
# TOP_N_KEYWORDS = 15 # 상위 몇개의 단어를 선택할지(추가적으로 검색할 단어 선정 과정에서)
# MY_STOPWORDS = ['don', 'like', 'nutrachamps', 'korean', 'panax', "ginseng", 've']
# # 일반적으로 추가해서 보려는 단어
# #   1. 리뷰의 변화를 나타내는 단어 : update, edit
# #   2. 재구매를 나타내는 표현 : 'second', 'again', 'reorder', 'another' 등의 단어들을 추가하는 것도 가능.
# SELF_SEARCH_WORDS = ['update', 'edit', 'again']  

# # 드라이버 시작
# driver = start_driver()
# driver.get(INITIAL_URL)

# # CAPTCHA 처리 + 로그인 등 페이지로 이동하는 작업 수행
# print("브라우저에서 CAPTCHA를 해결해주세요.")
# input("리뷰 페이지가 보이면 엔터를 누르세요...")

# # 최초 데이터 수집
# print("STEP 1: 최초 리뷰 데이터 수집 중...")
# initial_reviews = []
# current_page = 1

# while current_page <= INITIAL_MAX_PAGES:
#     print(f"  Initial Page {current_page}...")
#     page_data = collect_reviews_from_current_page(driver, keyword="Initial_Crawl")
#     initial_reviews.extend(page_data)
    
#     # 다음 페이지 이동
#     try:
#         next_button = driver.find_element(By.XPATH, "//li[@class='a-last']/a")
#         next_button.click()
#         time.sleep(4)
#         current_page += 1
#     except:
#         break

# df_initial = pd.DataFrame(initial_reviews)
# print(f"초기 수집 완료: {len(df_initial)}개 리뷰")


# # 데이터 분석 및 추가 키워드 추출
# print("\nSTEP 2: 텍스트 분석 및 추가 키워드 추출 중...")

# extracted_keywords = []

# if not df_initial.empty:
#     extracted_keywords = get_combined_keywords(
#         text_data=df_initial['리뷰 내용'],
#         top_n=TOP_N_KEYWORDS,
#         words_list=MY_STOPWORDS
#     )
#     print(f"추출된 키워드: {extracted_keywords}")

# if SELF_SEARCH_WORDS:
#     print(f'사용자 지정 단어 추가 : {SELF_SEARCH_WORDS}')
#     extracted_keywords.extend(SELF_SEARCH_WORDS)

# extracted_keywords = list(set(extracted_keywords))
# print(f"최종 검색할 키워드 리스트: {extracted_keywords}")


# # 추가 키워드로 2차 크롤링 (Concat을 위해 List에 모음)
# print("\nSTEP 3: 추출된 키워드로 상세 크롤링 시작...")
# additional_reviews = []

# # 각 키워드별 크롤링 수행
# for keyword in extracted_keywords:
#     kw_data = crawl_amazon_keyword(driver, keyword, max_pages=KEYWORD_MAX_PAGES)
#     additional_reviews.extend(kw_data)

# driver.quit()

# # 데이터 병합(최초 데이터 + 키워드 데이터), 중복 제거 및 최종 저장
# print("\nSTEP 4: 데이터 병합 및 저장 중...")

# df_additional = pd.DataFrame(additional_reviews)

# if not df_additional.empty:
#     print(f"추가 수집된 데이터: {len(df_additional)}개")
#     final_df = pd.concat([df_initial, df_additional], ignore_index=True)
# else:
#     print("추가 수집된 데이터가 없습니다.")
#     final_df = df_initial

# initial_len = len(final_df)
# final_df.drop_duplicates(subset=['작성자', '리뷰 내용'], keep='first', inplace=True)
# final_df = split_option(final_df, '옵션')
# print(f"중복 제거: {initial_len - len(final_df)}건 삭제됨.")

# output_filename = f"amazon_reviews_{PRODUCT_NAME}_final.csv"
# final_df.to_csv(output_filename, index=False, encoding='utf-8-sig')

# print(f"\n총 {len(final_df)}개의 리뷰가 '{output_filename}'에 저장되었습니다.")

브라우저에서 CAPTCHA를 해결해주세요.
STEP 1: 최초 리뷰 데이터 수집 중...
  Initial Page 1...
  Initial Page 2...
  Initial Page 3...
  Initial Page 4...
  Initial Page 5...
  Initial Page 6...
  Initial Page 7...
  Initial Page 8...
  Initial Page 9...
  Initial Page 10...
초기 수집 완료: 100개 리뷰

STEP 2: 텍스트 분석 및 추가 키워드 추출 중...
{'taking', 've', 'day', 'product', 'energy', 'good', 'great'}
추출된 키워드: ['taking', 've', 'day', 'product', 'energy', 'good', 'great']
최종 검색할 키워드 리스트: ['taking', 've', 'day', 'product', 'energy', 'good', 'great']

STEP 3: 추출된 키워드로 상세 크롤링 시작...

=== 키워드 검색 시작: 'taking' ===
  [Keyword: taking] Page 1...
  [Keyword: taking] Page 2...
  [Keyword: taking] Page 3...
  [Keyword: taking] Page 4...
  [Keyword: taking] Page 5...

=== 키워드 검색 시작: 've' ===
  [Keyword: ve] Page 1...

=== 키워드 검색 시작: 'day' ===
  [Keyword: day] Page 1...
  [Keyword: day] Page 2...
  [Keyword: day] Page 3...
  [Keyword: day] Page 4...
  [Keyword: day] Page 5...

=== 키워드 검색 시작: 'product' ===
  [Keyword: product] Page 1...
  [K